# 07 Open Data Cube: What It Adds and When to Use It

**Tutorial:** Building Geospatial Data Cubes for Earth Data Science  
**Focus:** Pine Ridge (Oglala Lakota Nation) 
**Prerequisites:** Notebooks 00–06  

## What Is Open Data Cube?
In notebooks 02–06, you built data cubes manually: fetch data, build
a DataArray, align coordinates, merge variables, save as NetCDF.
That approach works well for a single location and a few variables.

Open Data Cube (ODC) is a platform that automates this infrastructure
for large collections of satellite data:

| What you did manually | What ODC does automatically |
|---|---|
| Fetch tiles from APIs | Index data stored locally or in cloud storage |
| Align coordinates | Reproject and align on-the-fly at query time |
| Merge time steps | Assemble requested time range from indexed files |
| Track provenance | Store metadata alongside data in a database |
| Save to NetCDF | Read/write Zarr, GeoTIFF, NetCDF natively |

The mental model is the same, we use xarray datasets with named dimensions
and coordinates. ODC just handles the data management layer so you
can focus on analysis.

## When Do You Need ODC?
| Situation | Approach |
|---|---|
| Single location, few years, learning | Manual xarray |
| Multiple sites, many variables, research | Manual xarray and an organized file structure |
| Full satellite coverage, production analysis | Open Data Cube |
| Continental/global scale | ODC and cloud infrastructure (Cyverse) |

For most Tribal college projects, the manual xarray approach is
sufficient. ODC becomes valuable when you need full spatial coverage
at 250m or 30m resolution, for example, analyzing the entire Pine
Ridge reservation at Landsat resolution (30m) over 30 years.

## This Notebook
This notebook is **conceptual and exploratory** rather than
a full ODC setup guide. We:
1. Explain the ODC architecture and how it relates to what you have built
2. Show an ODC query side-by-side with the equivalent xarray query
3. Point to existing ODC deployments you can access without setup
4. Demonstrate how to access the Digital Earth Australia ODC sandbox
   (a free, no-setup ODC deployment useful for learning)

## Part 1: The ODC Architecture

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from src.cube_utils import (
    fetch_ndvi_timeseries,
    timeseries_to_dataarray,
    compute_growing_season_mean,
    CACHE_DIR,
)

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

print("Setup complete.")

## Part 2: Comparing xarray and ODC 
The code you write to query an ODC is nearly identical to the xarray
code you have been writing. The difference is where the data comes
from, is it from a managed index or an API call?

In [ ]:
# What you have been doing: manual xarray 

# Step 1: Fetch data from an API (manual)
# Step 2: Build an xarray DataArray (manual)
# Step 3: Select a time range (manual)

ndvi_df = fetch_ndvi_timeseries(
    lat=43.35, lon=-102.09,
    start_year=2010, end_year=2015,
    site_name="pine_ridge",
)
ndvi_da = timeseries_to_dataarray(ndvi_df, name="ndvi")
ndvi_subset = ndvi_da.sel(time=slice("2012-01-01", "2012-12-31"))

print("Manual xarray query result:")
print(ndvi_subset)
print(f"\n{len(ndvi_subset)} observations for 2012")

In [ ]:
# What the equivalent ODC query looks like
# This cell shows the code structure, but it won't run without a live ODC
# instance, but you can see the parallel to what you have been doing.

ODC_EQUIVALENT = '''
import datacube

# Connect to an ODC instance (local or cloud)
dc = datacube.Datacube()

# Query using same parameters as our manual approach,
# but ODC handles fetching, aligning, and assembling
ndvi_subset = dc.load(
    product   = "ls8_nbar_scene",     # indexed dataset name
    x         = (-102.5, -101.5),     # longitude range
    y         = (43.0,   44.0),       # latitude range
    time      = ("2012-01-01", "2012-12-31"),
    output_crs = "EPSG:4326",
    resolution = (-0.0002, 0.0002),   # ~25m
    measurements = ["nir", "red"],    # bands to load
)

# The result is an xarray Dataset with the same structure as what you built
print(ndvi_subset)     # xarray Dataset with time, lat, lon dimensions
print(type(ndvi_subset))  # <class 'xarray.core.dataset.Dataset'>

# Compute NDVI from the loaded bands 
ndvi = (ndvi_subset.nir - ndvi_subset.red) / \
       (ndvi_subset.nir + ndvi_subset.red)
'''

print("ODC equivalent query (illustrative only (requires live ODC)):")
print(ODC_EQUIVALENT)
print()
print("Observation: the dc.load() result is an xarray Dataset.")
print("Every analysis pattern from notebooks 03–06 works identically")
print("on ODC-loaded data as on our manually built cubes.")

## Part 3: ODC Components
Understanding the architecture helps you know what ODC is managing
on your behalf.

In [ ]:
# Print a clear explanation of ODC components and how they relate
# to what we have built manually

components = {
    "Product definition": (
        "A YAML file describing a dataset type: what satellite it came from, "
        "what bands/variables it contains, what CRS and resolution it uses. "
        "Equivalent to the attrs dict we put on our DataArrays."
    ),
    "Dataset document": (
        "A YAML/JSON file for each individual scene or tile: exact date, "
        "location, file path, quality flags. Like a detailed provenance "
        "record for one observation (IEEE 2890-2025 aligned)."
    ),
    "Index (database)": (
        "A PostgreSQL database that stores all dataset documents and enables "
        "fast spatial and temporal queries. When you call dc.load(), ODC "
        "queries the index to find which files cover your space/time request."
    ),
    "dc.load()": (
        "The main query function. Finds relevant files via the index, reads "
        "them, reprojects and resamples to your requested CRS and resolution, "
        "and returns an xarray Dataset. This replaces our manual "
        "fetch_ndvi_timeseries() to timeseries_to_dataarray() pipeline."
    ),
    "xarray Dataset": (
        "The output of dc.load() is identical in structure to what you have "
        "been building. All the .sel(), .where(), .groupby(), .resample() "
        "operations from notebooks 03–06 work exactly the same way."
    ),
}

print("OPEN DATA CUBE ARCHITECTURE")
print("=" * 65)
for component, description in components.items():
    print(f"\n{component}:")
    # Wrap text at 65 chars
    words, line = description.split(), ""
    for word in words:
        if len(line) + len(word) + 1 > 63:
            print(f"  {line}")
            line = word
        else:
            line = f"{line} {word}".strip()
    if line:
        print(f"  {line}")

## Part 4: Existing ODC Deployments
You do not need to set up your own ODC to start using one.
Several free deployments exist that you can query directly.

In [ ]:
deployments = [
    {
        "name":    "Digital Earth Australia (DEA)",
        "url":     "https://www.dea.ga.gov.au/",
        "sandbox": "https://app.sandbox.dea.ga.gov.au/",
        "data":    "Landsat, Sentinel-2, SAR for Australia",
        "access":  "Free sandbox with JupyterHub, no install required",
        "note":    "Best free ODC learning environment. Australian data only.",
    },
    {
        "name":    "Digital Earth Africa",
        "url":     "https://www.digitalearthafrica.org/",
        "sandbox": "https://sandbox.digitalearth.africa/",
        "data":    "Landsat, Sentinel-2, SAR — Africa",
        "access":  "Free sandbox, no install required",
        "note":    "African data, but ODC query syntax is identical.",
    },
    {
        "name":    "USGS Landsat on AWS",
        "url":     "https://registry.opendata.aws/usgs-landsat/",
        "sandbox": "N/A — requires AWS account",
        "data":    "Landsat Collection 2 with global CONUS available",
        "access":  "ODC can index and query via STAC catalog",
        "note":    "Requires local ODC setup. Best path for US/Tribal lands data.",
    },
    {
        "name":    "Microsoft Planetary Computer",
        "url":     "https://planetarycomputer.microsoft.com/",
        "sandbox": "https://planetarycomputer.microsoft.com/docs/overview/about/",
        "data":    "Landsat, Sentinel, MODIS, NAIP (global)",
        "access":  "Free JupyterHub, uses STAC and odc-stac (ODC-compatible)",
        "note":    "Excellent for CONUS analysis. Uses odc-stac which is ODC-compatible.",
    },
]

print("ACCESSIBLE ODC DEPLOYMENTS")
for d in deployments:
    print(f"\n{d['name']}")
    print(f"  Data    : {d['data']}")
    print(f"  Access  : {d['access']}")
    print(f"  Sandbox : {d['sandbox']}")
    print(f"  Note    : {d['note']}")

print()
print("RECOMMENDATION FOR THIS TUTORIAL:")
print("  Start with Digital Earth Australia sandbox (free, no setup),")
print("  and the ODC query syntax is identical to what you would use")
print("  for US data. Learn the interface there, then apply it to")
print("  Planetary Computer for CONUS/South Dakota data.")

## Part 5: Running the Same Analysis Patterns on ODC Data
To show that everything you learned applies directly, here is how each
notebook 06 pattern translates to an ODC workflow.

In [ ]:
# This cell shows the translation of each pattern and is illustrative only

translations = [
    (
        "Load data",
        # Manual
        "df = fetch_ndvi_timeseries(lat, lon, 2000, 2023)\n"
        "da = timeseries_to_dataarray(df)",
        # ODC
        "ds = dc.load(product='ga_ls8c_nbart_gm_cyear_3',\n"
        "             x=(-102.5,-101.5), y=(43.0,44.0),\n"
        "             time=('2000','2023'))",
    ),
    (
        "Time slice",
        "da.sel(time=slice('2012-01-01','2012-12-31'))",
        "ds.sel(time=slice('2012-01-01','2012-12-31'))",
    ),
    (
        "Annual mean",
        "da.groupby('time.year').mean()",
        "ds.groupby('time.year').mean()   # identical",
    ),
    (
        "Anomaly",
        "da - da.mean()",
        "ds - ds.mean()   # identical",
    ),
    (
        "Spatial clip",
        "da.rio.clip(boundary.geometry, crs=...)",
        "dc.load(..., geopolygon=boundary_geom)\n"
        "# ODC clips at load time, no post-load clip needed",
    ),
    (
        "Save to NetCDF",
        "ds.to_netcdf('cube.nc')",
        "ds.to_netcdf('cube.nc')   # identical",
    ),
]

print("ANALYSIS PATTERN TRANSLATION: Manual xarray to ODC")
for name, manual, odc in translations:
    print(f"\n{name}:")
    print(f"  Manual : {manual.splitlines()[0]}")
    if len(manual.splitlines()) > 1:
        for line in manual.splitlines()[1:]:
            print(f"           {line}")
    print(f"  ODC    : {odc.splitlines()[0]}")
    if len(odc.splitlines()) > 1:
        for line in odc.splitlines()[1:]:
            print(f"           {line}")

## Part 6: Setting Up a Local ODC (Optional)
If you want to run your own ODC instance against US satellite data,
here is the minimum path to get there.

In [ ]:
SETUP_GUIDE = """
LOCAL ODC SETUP — MINIMUM PATH

1. Install ODC (already in environment.yml for this tutorial):
   conda install datacube -c conda-forge

2. Initialize a database:
   # Requires PostgreSQL
   createdb datacube
   datacube -v system init

3. Add a product definition (describes what data you will index):
   datacube product add product_definition.yaml

4. Index datasets (tells ODC where your files are):
   # Option A: Index from a STAC catalog (recommended for US data)
   pip install odc-stac
   # Then use odc-stac to load directly from Microsoft Planetary Computer
   # or Cyverse with no local indexing required

   # Option B: Index local files
   datacube dataset add dataset_document.yaml

5. Load and analyze:
   import datacube
   dc = datacube.Datacube()
   ds = dc.load(product='your_product', x=..., y=..., time=...)

RECOMMENDED SHORTCUT FOR US DATA:
   Use odc-stac to query Planetary Computer directly, no local database needed:
   
   import odc.stac
   import pystac_client
   
   catalog = pystac_client.Client.open(
       'https://planetarycomputer.microsoft.com/api/stac/v1'
   )
   items = catalog.search(
       collections=['landsat-c2-l2'],
       bbox=[-102.5, 43.0, -101.5, 44.0],
       datetime='2012-01-01/2012-12-31',
   ).item_collection()
   
   ds = odc.stac.load(items, crs='EPSG:4326', resolution=0.0002)
   # ds is an xarray Dataset, same as dc.load() output
"""

print(SETUP_GUIDE)

---
## Part 7: ODC and Data Sovereignty

ODC introduces infrastructure considerations that matter for Tribal
data governance.

In [ ]:
print("""
ODC AND TRIBAL DATA SOVEREIGNTY

When you use a public ODC deployment (Planetary Computer, DEA sandbox),
you are running analysis on someone else's infrastructure. For analysis
using only public federal data (MODIS, Landsat), this is generally fine.

If your ODC workflow involves Tribal-collected data such as pasture condition
scores, well levels, community observations, etc., the infrastructure choice
matters for data sovereignty:

PUBLIC CLOUD ODC (Planetary Computer, AWS)
  No setup cost, large data capacity
  Data stored on third-party infrastructure
  Requires trust in cloud provider's data handling
  Appropriate for: public data analysis, research without Tribal data

LOCAL ODC (on Tribal server or Tribal College infrastructure)
  Data stays under Tribal control
  Aligns with OCAP® (Possession means physical control)
  Can index both public and Tribal-collected data
  Requires local server setup and maintenance
  Appropriate for: operational programs, Tribal data integration

HYBRID (public data in cloud, Tribal data local)
  Public satellite data accessed from cloud for scale
  Tribal observational data stays local
  Analysis merges both without moving Tribal data to cloud
  Best practice for production Tribal environmental monitoring

The CARE Principles apply directly here:
  A: Authority to Control: Tribal Nations should control where
      their environmental data lives and who can access it.
  C: Collective Benefit: Infrastructure choices should serve
      the community's long-term capacity, not just current convenience.
""")

## Summary: When to Use What
| Tool | Best for | Limitations |
|---|---|---|
| **Manual xarray** (this tutorial) | Learning, single site, custom APIs | Manual data management |
| **ODC and local files** | Full spatial coverage, research | Requires setup and storage |
| **odc-stac and Planetary Computer** | US satellite data at scale, no setup | Public cloud dependency |
| **ODC on Tribal infrastructure** | Production programs, Tribal data | Requires IT support |

## Discussion Questions
1. If the Oglala Lakota Nation wanted to build an environmental monitoring
   system that combined MODIS satellite data with Tribal-collected pasture
   condition scores, which ODC architecture would you recommend? What
   would need to be in place at the Tribal College or Nation's IT
   infrastructure to support it?

2. The Digital Earth Australia sandbox is a free way to practice ODC
   queries. The data is Australian, but the query syntax is identical
   to what you would use for Pine Ridge data. What would you explore
   first in the DEA sandbox using the analysis patterns from notebook 06?

3. ODC's provenance metadata capabilities align well with IEEE 2890-2025.
   If you were designing a Tribal environmental monitoring ODC deployment,
   what provenance fields would you require for every indexed dataset?

## Next Steps
- Explore DEA Sandbox: https://app.sandbox.dea.ga.gov.au/
  Run the analysis patterns from notebook 06 on Australian Landsat data.
  The syntax is identical; only the product name and coordinates change.

- Explore Planetary Computer:
  https://planetarycomputer.microsoft.com/
  US Landsat and Sentinel-2 data accessible via odc-stac.

- ODC documentation: https://datacube-core.readthedocs.io/

- odc-stac (ODC and STAC, no database required):
  https://odc-stac.readthedocs.io/

## Next Notebook
**08 Your Own Question:** A capstone or final project. Define a question about
the land that matters to you, identify the data you need, build
the cube, and produce one visualization that answers the question.